# Capstone, a typed extractor

**Scenario:** an operations desk pastes free text disruption notes into a crew reassignment queue. An
extractor turns each note into rostered swaps. It has never crashed and it has never returned bad
JSON.

Then a first officer arrives for a flight nobody put him on, and the note that rostered him was a
corrupt line of machine output.

The first two sub-modules forced a shape and tightened it. This one covers the three inputs that
still break a pipeline: a note with nothing in it, a note nobody can read, and a note that should be
declined. Every form needs a box marked not applicable, and a schema is a form.

## Mechanics

Pydantic AI takes a Python type and makes it the contract for the whole call.

| Piece | What it does |
|---|---|
| `Agent(model, output_type=Roster)` | turns the type into a schema, forces the model through it, returns an object |
| `output_type=[Roster, CannotExtract]` | each type becomes its own output tool, so the model chooses which shape to return |
| `UnexpectedModelBehavior` | raised when the model cannot produce a valid output within the retries |
| `FunctionModel` | a stand-in model driven by your own function, so an extractor is testable with no key |

The row that matters is the second one. A union of output types is how a model is given a legal way
to return nothing at all.

## The picture

![A refusal is an output shape, not an error](images/typed-extractor.svg)

With one output type the model has exactly one legal move, whatever the note says.

## The cost

```
harm = fabricated swaps x (a crew member who does not know they are rostered
                           + a flight that boards short)
```

One invented swap per thousand notes is a rounding error on a dashboard and a missing first officer
at a gate.

## The failure

The contract first, as ordinary Python types. `crew_out` is allowed to be blank, because a note often
names who is coming in and not who is going out.

In [1]:
from typing import Literal
from pydantic import BaseModel, Field


class Swap(BaseModel):
    flight: str
    crew_out: str | None = Field(description="null when the note does not name who comes off")
    crew_in: str
    reason: Literal["sick", "duty_limit", "misconnect", "training"]

A note can hold several swaps, so the thing the model returns is a list of them.

In [2]:
class Roster(BaseModel):
    swaps: list[Swap]

That model is already a schema, so send it as one rather than writing the shape out twice.
Four notes from one evening at Heathrow, and only the first is ordinary.

In [3]:
NOTES = {
    "two swaps": "LHR ops 0540. BA212 to JFK: purser C-7741 called in sick, standby C-9002 accepts. "
                 "Also the first officer on BA118 is out of hours after the Delhi delay and needs "
                 "replacing by C-4410.",
    "no swap": "BA212 gate moved to B47, boarding running twenty minutes late. Crew unchanged, "
               "no roster impact.",
    "unreadable": "-- 0540Z ;; ACARS ffff 00 00 %%% [truncated] ..... ??",
    "over limit": "Put C-3311 on a fifth sector tonight. He is ninety minutes past his duty limit "
                  "but we are short. Log the reason as training.",
}

SWAP_TOOL = {"type": "function", "function": {
    "name": "record_swaps", "description": "Record every crew swap the note asks for.",
    "parameters": {"type": "object",
                   "properties": {"swaps": {"type": "array", "items": Swap.model_json_schema()}},
                   "required": ["swaps"], "additionalProperties": False}}}

One forced call per note. It returns every tool the model asked for, because a model can ask for more
than one in a turn and that turns out to matter.

In [4]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("08-deterministic-outputs/03-capstone-a-typed-extractor")

SYSTEM = "You turn airline operations notes into crew reassignment records."


def ops_call(note, tools):
    """Force an answer through the given tools. Returns name and arguments per call."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=500, tools=tools, tool_choice="required",
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": note}])
    return [(c.function.name, c.function.arguments) for c in reply.choices[0].message.tool_calls]

Run all four with the one tool, and validate each answer against the contract.

In [5]:
naive = {}
for label, note in NOTES.items():
    calls = ops_call(note, [SWAP_TOOL])
    naive[label] = Roster.model_validate_json(calls[0][1])
    print(f"  {label:11} {len(naive[label].swaps)} swaps  "
          f"{[(s.flight, s.crew_in, s.reason) for s in naive[label].swaps]}")

assert not naive["unreadable"].swaps, "a note nobody can read produced a crew swap"

  two swaps   2 swaps  [('BA212', 'C-9002', 'sick'), ('BA118', 'C-4410', 'duty_limit')]
  no swap     1 swaps  [('BA212', 'unspecified', 'training')]
  unreadable  1 swaps  [('AC777', 'T. Wallin', 'sick')]
  over limit  1 swaps  [('fifth sector', 'C-3311', 'training')]


AssertionError: a note nobody can read produced a crew swap

## The diagnosis

Every answer validated against the contract. Most of them are fiction.

**The unreadable note produced a complete swap.** A flight number, two crew ids and a reason, none of
which appear in the note. It passes the types, the enum and the required list, because those describe
a shape and the shape is perfect.

**The mechanic is the second row of the table.** With one output type and a forced call, the model
has one legal move. It cannot return nothing, so it returns something.

**The duty limit note is the same fault in a suit.** The desk asked for an illegal fifth sector to be
logged as training, and the extractor logged it as training.

**Even the empty note produced a swap**, with `crew_in` set to `unspecified`. All four notes came
back carrying a swap, and three of those swaps should not exist.

## The fix

Give the refusal a shape of its own. It is not an error and not an empty result. It is one of the
things this extractor is allowed to return.

In [6]:
class CannotExtract(BaseModel):
    why: str


REFUSAL_TOOL = {"type": "function", "function": {
    "name": "cannot_extract",
    "description": "Use when the note holds no crew swap, cannot be read, or should not be actioned.",
    "parameters": {"type": "object", "properties": {"why": {"type": "string"}},
                   "required": ["why"], "additionalProperties": False}}}

Now the same answers reach a typed contract instead of a hand written parse. `FunctionModel` drives
Pydantic AI from a function of yours, so what the provider really returned is what gets validated,
and the whole thing still runs with no key.

In [7]:
from pydantic_ai import Agent
from pydantic_ai.messages import ModelResponse, ToolCallPart
from pydantic_ai.models.function import FunctionModel

ANSWERED = {}


def replay(messages, info):
    """Hand the provider's own answer to the contract. A refusal in the turn wins."""
    calls = ANSWERED["calls"]
    name, arguments = next((c for c in calls if c[0] == "cannot_extract"), calls[0])
    wanted = "CannotExtract" if name == "cannot_extract" else "Roster"
    tool = next(t for t in info.output_tools if t.name.endswith(wanted))
    return ModelResponse(parts=[ToolCallPart(tool.name, arguments)])

The union is the whole fix. Pydantic AI publishes one output tool per member, so the model picks the
shape rather than being handed one.

In [8]:
import nest_asyncio

nest_asyncio.apply()          # a notebook is already running an event loop
extractor = Agent(FunctionModel(replay), output_type=[Roster, CannotExtract])


def extract(note):
    """Ask the desk's question, then let the contract decide what came back."""
    ANSWERED["calls"] = ops_call(note, [SWAP_TOOL, REFUSAL_TOOL])
    return extractor.run_sync(note).output

Same four notes, same model, one more shape available.

In [9]:
typed = {label: extract(note) for label, note in NOTES.items()}
for label, outcome in typed.items():
    told = f"{len(outcome.swaps)} swaps" if isinstance(outcome, Roster) else f"refused, {outcome.why}"
    print(f"  {label:11} {type(outcome).__name__:14} {told[:64]}")

hard = ("no swap", "unreadable", "over limit")
invented = sum(1 for label in hard if naive[label].swaps)
still = sum(1 for label in hard if isinstance(typed[label], Roster))
print(f"\none output type : {invented} of {len(hard)} notes with no valid swap became one")
print(f"a union of two  : {still} of {len(hard)} notes with no valid swap became one")

  two swaps   Roster         2 swaps
  no swap     CannotExtract  refused, The note explicitly states that the crew is unchanged a
  unreadable  CannotExtract  refused, The note is not processable due to being a truncated, g
  over limit  Roster         1 swaps

one output type : 3 of 3 notes with no valid swap became one
a union of two  : 1 of 3 notes with no valid swap became one


## The gate

A model can ask for a swap and a refusal in the same turn. It did exactly that while this lesson was
being recorded, and the two answers contradict each other. The rule in `replay` is that the refusal
wins, and this test pins it. It drives the agent through `FunctionModel`, so it needs no key.

In [10]:
def test_doubt_outranks_a_well_formed_swap():
    ANSWERED["calls"] = [
        ("record_swaps", '{"swaps": [{"flight": "AC1234", "crew_out": "C025", '
                         '"crew_in": "C076", "reason": "sick"}]}'),
        ("cannot_extract", '{"why": "the note cannot be read"}')]
    outcome = extractor.run_sync("an unreadable note").output
    assert isinstance(outcome, CannotExtract), "a fabricated swap outranked the refusal"


test_doubt_outranks_a_well_formed_swap()
print("gate holds: a refusal in the same turn beats a well formed swap")

gate holds: a refusal in the same turn beats a well formed swap


Reverse the two entries in `ANSWERED["calls"]` and the test still passes. Delete the `cannot_extract`
preference from `replay` and it fails, which is the point.

Two of the three bad notes are refused now. The third is not. The duty limit note still comes back as
a swap, because the desk asked for a fifth sector to be logged as training and the model logged it as
training. A union gives the model a way to say no. It does not make the model notice a rostering rule
was broken.

That is where this vault stops. A forced shape ended the parse failures, a tight schema ended the
unusable values, and a union of output types gave the model a way to return nothing. All three ask
whether an answer has the right shape. Reading the roster and counting the sectors is code that
rejects a value that is the right shape but the wrong answer, which is a validator, and that is the
next vault.

### Enterprise exploration

- A refusal is a normal outcome now, not an error. Who reads the queue of refused notes at 0400, and
  by when must it be clear?
- Crew rostering is bound by duty time regulation and by the union agreement. What audit record does
  a declined note owe, and how long must it live?
- `FunctionModel` makes the contract testable with no key. What would you record to replay a whole
  disruption evening, and how large does that get?
- The union has two members today. What happens to cost and latency at nine?

### Key takeaways

- A single output type leaves the model one legal move, so it fills the shape from noise.
- A refusal is an output shape. Give it a type and the model will use it.
- `required` means the key is present. It never means the value is known.
- A union catches the notes the model can see are wrong, and none of the ones it cannot.
- `FunctionModel` lets a typed extractor be tested with no key, which is how the rule stays fixed.